# Train results — R1

Reproduce R1 training metrics from the saved `subset_train-r1` run. No model requests are made.


## 1. Configuration and offline analysis
Run from the repository root. This regenerates the summary CSVs.


In [ ]:
from pathlib import Path
import subprocess, sys
ROOT=Path.cwd()
while not (ROOT/'scripts/analyze_repair_strategies.py').exists() and ROOT!=ROOT.parent:
    ROOT=ROOT.parent
assert (ROOT/'scripts/analyze_repair_strategies.py').exists(), 'Run notebook inside the repository'
RUN_DIR=ROOT/'experiments/subset_train-r1'
OUTPUT=ROOT/'train_result_analysis/analysis_output'
assert (RUN_DIR/'attempts.jsonl').exists()
subprocess.run([sys.executable, str(ROOT/'scripts/analyze_repair_strategies.py'), '--run', f'R1={RUN_DIR}', '--output-dir', str(OUTPUT), '--no-figures'], cwd=ROOT, check=True)


## 2. Load summaries and calculate eligible-denominator rates
Localisation success is computed using eligible attempts, not all attempts.


In [ ]:
import csv
def load(path):
    with open(path,encoding='utf-8-sig',newline='') as f:
        return {r['metric']:r for r in csv.DictReader(f)}
DATA=OUTPUT/'R1'
overall=load(DATA/'overall_summary.csv')
mech=load(DATA/'mechanism_summary.csv')
def iv(k): return int(float(overall[k]['value']))
def count_rate(k):
    r=mech[k]; c=int(float(r['count'])); n=int(float(r['valid_n']))
    return f'{c} / {n} ({float(r["rate"])*100:.2f}%)'
p=mech['personal_localization_success']; c=mech['commonsense_localization_success']
success=int(float(p['count']))+int(float(c['count']))
eligible=int(float(p['valid_n']))+int(float(c['valid_n']))
print('Overall localisation:',f'{success}/{eligible} ({100*success/eligible:.2f}%)')


## 3. Export the Excel table
Metrics are derived from the saved summary CSVs. Additional detailed evidence remains in `localization.csv` and `ANALYSIS.md`.


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
rows=[('General','Queries',iv('declared_queries')),('General','Repair attempts',iv('total_attempts')),('General','Personal repair attempts',iv('personal_attempts')),('General','Commonsense repair attempts',iv('commonsense_attempts')),('Localisation','Overall localisation success',f'{success} / {eligible} ({100*success/eligible:.2f}%)')]
for category,label,key in [('Localisation','Personal localisation success','personal_localization_success'),('Localisation','Commonsense localisation success','commonsense_localization_success'),('Replacement','Personal invalid replacement','personal_invalid_replacement_after_slot'),('Replacement','Commonsense invalid replacement','commonsense_invalid_replacement_after_slot')]: rows.append((category,label,count_rate(key)))
unchanged=iv('unchanged_confirmed'); attempts=iv('total_attempts')
rows.append(('No progress','Confirmed unchanged attempts',f'{unchanged} / {attempts} ({100*unchanged/attempts:.2f}%)'))
for category,label,key in [('Repair outcome','Personal fixes after slot selection','personal_fixed_constraint_after_selection'),('Repair outcome','Commonsense fixes after slot selection','commonsense_fixed_constraint_after_selection'),('Repair outcome','Measured one-slot fixes','successful_measured_one_slot_repair')]: rows.append((category,label,count_rate(key)))
wb=Workbook(); ws=wb.active; ws.title='R1 train results'; ws.append(('Category','Metric','R1 TRAIN result'))
for row in rows: ws.append(row)
ws.freeze_panes='A2'; ws.auto_filter.ref=ws.dimensions
for cell in ws[1]: cell.font=Font(color='FFFFFF',bold=True); cell.fill=PatternFill('solid',fgColor='183153')
for row in ws.iter_rows(min_row=2):
    for cell in row: cell.alignment=Alignment(vertical='top',wrap_text=True)
    if row[0].row%2==0:
        for cell in row: cell.fill=PatternFill('solid',fgColor='EEF3F8')
for col,width in [('A',19),('B',49),('C',68)]: ws.column_dimensions[col].width=width
out=ROOT/'train_result_analysis/Train_results_R1.xlsx'; wb.save(out); print('Wrote',out)
